# BƯỚC 1: TRÍCH XUẤT DỮ LIỆU (EXTRACTION)

In [58]:
import logging
import traceback
import subprocess
import openpyxl
import pdfplumber
import pytesseract

try:
    from docling.document_converter import DocumentConverter as DoclingConverter
except (ImportError, RuntimeError, Exception) as e:
    print(f"⚠️ Cảnh báo: Docling không thể load do xung đột thư viện ({type(e).__name__}: {e}).")
    DoclingConverter = None

from pdf2image import convert_from_path
from PIL import Image
from docx import Document as DocxDocument

import os
os.environ["CUDA_VISIBLE_DEVICES"] = "1"  

logging.basicConfig(level=logging.INFO, format="%(levelname)s - %(message)s")
logger = logging.getLogger(__name__)

SUPPORTED_EXT = {".pdf", ".docx", ".doc", ".xlsx", ".xls", ".pptx", ".ppt", ".md", ".txt"}

PDF_OCR_STYLE = {
    "prompt": "<image>\n<|grounding|>Convert the document to markdown. ",
    "base_size": 1024,
    "image_size": 640,
    "crop_mode": True,
    "test_compress": True,
    "save_results": True,
}


In [59]:
from pathlib import Path

input_file_path = "/home/trung-ai/chatbot_hcns/create_documents/doc/241025_E_QUY_TRÌNH_DỰ_ÁN.pdf"
input_path = Path(input_file_path)

# Cấu hình các thư mục lưu trữ theo yêu cầu
md_files_dir = Path("/home/trung-ai/chatbot_hcns/create_documents/md_files")
md_final_files_dir = Path("/home/trung-ai/chatbot_hcns/create_documents/md_final_files")

# Tạo thư mục nếu chưa tồn tại
md_files_dir.mkdir(parents=True, exist_ok=True)
md_final_files_dir.mkdir(parents=True, exist_ok=True)

# Cấu hình đường dẫn file
extracted_markdown_path = str(md_files_dir / f"{input_path.stem}.md")
cleaned_markdown_path = str(md_final_files_dir / f"{input_path.stem}_cleaned.md")


print("⚙️ CẤU HÌNH ĐƯỜNG DẪN PIPELINE MỚI:")
print(f"1. PDF Input : {input_path}")
print(f"2. Raw MD    : {extracted_markdown_path}")
print(f"3. Clean MD  : {cleaned_markdown_path}")

⚙️ CẤU HÌNH ĐƯỜNG DẪN PIPELINE MỚI:
1. PDF Input : /home/trung-ai/chatbot_hcns/create_documents/doc/241025_E_QUY_TRÌNH_DỰ_ÁN.pdf
2. Raw MD    : /home/trung-ai/chatbot_hcns/create_documents/md_files/241025_E_QUY_TRÌNH_DỰ_ÁN.md
3. Clean MD  : /home/trung-ai/chatbot_hcns/create_documents/md_final_files/241025_E_QUY_TRÌNH_DỰ_ÁN_cleaned.md


In [60]:

# # ── CELL TẠM: Gán thẳng đường dẫn file (bỏ qua BƯỚC 1 & 2) ──
# from pathlib import Path

# # Nếu bạn muốn gán thủ công một file đã clean
# cleaned_markdown_path = "/home/trung-ai/chatbot_hcns/create_documents/md_final_files/overview_data_cleaned.md" 

# _cleaned = Path(cleaned_markdown_path)
# json_chunking_dir = Path("/home/trung-ai/chatbot_hcns/create_documents/json_chunking")
# json_chunking_dir.mkdir(parents=True, exist_ok=True)

# staging_chunks_json_path = str(json_chunking_dir / f"{_cleaned.stem.replace('_cleaned', '')}_chunks.json")

# print(f"📄 Cleaned MD  : {cleaned_markdown_path}")
# print(f"📁 Chunks JSON : {staging_chunks_json_path}")


In [61]:
from transformers import AutoTokenizer, AutoModel
import torch
import os

model_id = "/home/trung-ai/chatbot_hcns/weights/DeepSeek-OCR-bnb-4bit-NF4"

print(f"⏳ Đang tải Tokenizer từ {model_id}...")
# Patch Llama compatibility mapping để chạy ổn định
try:
    from transformers.models.llama import modeling_llama
    if not hasattr(modeling_llama, "LlamaFlashAttention2"):
        setattr(modeling_llama, "LlamaFlashAttention2", modeling_llama.LlamaAttention)
except ImportError:
    pass

tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)

print(f"⏳ Đang tải Model OCR từ {model_id} (bfloat16)...")
# Cấu hình bfloat16 và device_map="auto" đồng bộ với convert_md.py chuẩn
model = AutoModel.from_pretrained(
    model_id,
    trust_remote_code=True,
    use_safetensors=True,
    device_map="auto",
    torch_dtype=torch.bfloat16,
).eval()

print("⏳ Đang khởi tạo Docling Converter...")
if DoclingConverter:
    try:
        docling_converter = DoclingConverter()
    except Exception as e:
        print(f"⚠️ Không thể khởi tạo Docling: {e}")
        docling_converter = None
else:
    docling_converter = None
    print("ℹ️ Docling không khả dụng.")

print("✅ Cấu hình hoàn tất!")


⏳ Đang tải Tokenizer từ /home/trung-ai/chatbot_hcns/weights/DeepSeek-OCR-bnb-4bit-NF4...


Unused kwargs: ['_load_in_4bit', '_load_in_8bit', 'quant_method']. These kwargs are not used in <class 'transformers.utils.quantization_config.BitsAndBytesConfig'>.


⏳ Đang tải Model OCR từ /home/trung-ai/chatbot_hcns/weights/DeepSeek-OCR-bnb-4bit-NF4 (bfloat16)...


INFO - We will use 90% of the memory on device 0 for storing the model, and 10% for the buffer to avoid OOM. You can set `max_memory` in to a higher value to use more memory (at your own risk).


⏳ Đang khởi tạo Docling Converter...
✅ Cấu hình hoàn tất!


In [62]:
def auto_rotate(image_path: Path) -> None:
    try:
        img = Image.open(image_path)
        osd = pytesseract.image_to_osd(img, output_type=pytesseract.Output.DICT)
        angle = osd.get("rotate", 0)
        if angle:
            img.rotate(-angle, expand=True).save(image_path)
    except Exception:
        pass

def is_pdf_scan(pdf_path: str) -> bool:
    try:
        total_chars = 0
        with pdfplumber.open(pdf_path) as pdf:
            for page in pdf.pages:
                text = page.extract_text()
                if text:
                    total_chars += len(text.strip())
                    if total_chars >= 100:
                        return False
        return True
    except Exception as exc:
        logger.warning("Unable to check PDF scan status: %s", exc)
        return True

def clean_ocr_noise(text: str) -> str:
    # 1. Loại bỏ các dòng Header/Footer lặp lại (Mã tài liệu, Ngày ban hành, Phiên bản)
    noise_patterns = [
        r"(?i)QUY ĐỊNH CHẤM CÔNG",
        r"(?i)Mã tài liệu:\s*HR-QĐ-\d+",
        r"(?i)Phiên bản:\s*\d+",
        r"(?i)Ngày ban hành:\s*\d{2}/\d{2}/\d{4}",
        r"(?i)Trang\s+\d+\s*/\s*\d+"
    ]
    for pattern in noise_patterns:
        text = re.sub(pattern, "", text)
    
    # 2. Loại bỏ bảng trống (thường là rác OCR hoặc header bảng bị tách)
    # Tìm tất cả các bảng <table>...</table>
    tables = re.findall(r"(<table.*?>.*?</table>)", text, flags=re.DOTALL | re.IGNORECASE)
    keywords = {"lần", "số yêu cầu", "sửa đổi", "nội dung", "ngày", "người", "revision", "date", "description", "author"}
    
    for table in tables:
        # Lấy nội dung text thuần trong các cell
        cells = re.findall(r"<t[dr].*?>(.*?)</t[dr]>", table, flags=re.DOTALL | re.IGNORECASE)
        plain_cells = [re.sub(r"<.*?>", "", c).strip() for c in cells]
        
        # Một bảng được coi là "trống" nếu không có cell nào chứa dữ liệu thực tế (ngoài các từ khóa header rác)
        has_real_data = any(c and not any(kw in c.lower() for kw in keywords) for c in plain_cells)
        if not has_real_data:
            text = text.replace(table, "")
            
    return text

import io
import contextlib
import re
import unicodedata


def _strip_diacritics(text: str) -> str:
    text = text.replace("Đ", "D").replace("đ", "d")
    nfkd = unicodedata.normalize("NFKD", text)
    return "".join(c for c in nfkd if not unicodedata.combining(c))


def _get_subtitle_heading(content: str) -> str:
    clean_content = re.sub(r'^[#\s]+', '', content).strip()
    
    normalized = _strip_diacritics(clean_content).lower()
    if re.search(r"chuong|phan", normalized):
        return f"## {clean_content}"
    elif re.search(r"dieu", normalized):
        return f"### {clean_content}"
    else:
        return f"#### {clean_content}"


def smart_parse_ocr_tags(text: str) -> str:
    parts = re.split(r'(<\|ref\|>.*?<\|/ref\|>)', text, flags=re.DOTALL)
    cleaned_blocks = []
    current_role = "text"

    for part in parts:
        part = part.strip()
        if not part: continue
            
        if part.startswith("<|ref|>"):
            role_match = re.search(r'<\|ref\|>(.*?)<\|/ref\|>', part)
            current_role = role_match.group(1).strip() if role_match else "text"
            continue
        
        content = re.sub(r'<\|det\|>.*?<\|/det\|>', '', part, flags=re.DOTALL).strip()
        if not content: continue
            
        if current_role == "sub_title":
            cleaned_blocks.append(_get_subtitle_heading(content))
        else:
            cleaned_blocks.append(content)

    return "\n\n".join(cleaned_blocks)


def process_pdf_scan(pdf_path: str, output_dir: str) -> str:
    pages = convert_from_path(pdf_path, dpi=300)
    os.makedirs(output_dir, exist_ok=True)
    output = []
    
    for i, page in enumerate(pages):
        img_path = Path(output_dir) / f"temp_{i}.png"
        try:
            page.save(img_path)
            auto_rotate(img_path)
            
            f = io.StringIO()
            with contextlib.redirect_stdout(f):
                model.infer(
                    tokenizer,
                    prompt=PDF_OCR_STYLE["prompt"],
                    image_file=str(img_path),
                    output_path=str(output_dir),
                    base_size=PDF_OCR_STYLE["base_size"],
                    image_size=PDF_OCR_STYLE["image_size"],
                    crop_mode=PDF_OCR_STYLE["crop_mode"],
                    save_results=True,
                    test_compress=True
                )
            
            full_log = f.getvalue()
            match = re.search(r'(<\|ref\|>.*?)={10,}', full_log, flags=re.DOTALL)
            if not match: continue
            
            raw_text_with_tags = match.group(1).strip()
            if raw_text_with_tags:
                parsed_text = smart_parse_ocr_tags(raw_text_with_tags)
                # Dọn sạch noise (Header/Footer/Empty Tables) ngay từng trang
                output.append(clean_ocr_noise(parsed_text))
            
        except Exception as e:
            print(f"⚠️ Lỗi tại trang {i+1}: {e}")
        finally:
            img_path.unlink(missing_ok=True)
            
    return "\n\n".join(output)

In [63]:

def convert_document(file_path: str, output_dir: str = "/tmp") -> str:
    """Hàm tổng điều phối việc chuyển đổi file theo logic convert_md.py"""
    file_path_str = str(file_path)
    ext = Path(file_path_str).suffix.lower()
    content = None
    
    try:
        if ext == ".doc":
            dest = Path(file_path_str).with_suffix(".docx")
            if not dest.exists():
                subprocess.run(
                    ["soffice", "--headless", "--convert-to", "docx", "--outdir", str(dest.parent), file_path_str],
                    check=True, capture_output=True
                )
            file_path_str = str(dest)
            ext = ".docx"

        if ext == ".pdf":
            # Nếu là scan, bốc DeepSeek-OCR ngay
            if is_pdf_scan(file_path_str):
                logger.info("Phát hiện PDF Scan, đang kích hoạt DeepSeek-OCR...")
                content = process_pdf_scan(file_path_str, output_dir)
            else:
                # Nếu là text, thử dùng Docling trước
                logger.info("Phát hiện PDF Text, thử chạy Docling...")
                if docling_converter:
                    try:
                        content = docling_converter.convert(file_path_str).document.export_to_markdown()
                    except Exception as e:
                        logger.warning(f"Docling lỗi ({e}), chuyển sang fallback DeepSeek-OCR...")
                        content = process_pdf_scan(file_path_str, output_dir)
                else:
                    logger.info("Docling không sẵn sàng, dùng DeepSeek-OCR...")
                    content = process_pdf_scan(file_path_str, output_dir)
        else:
            # Non-PDF files
            if docling_converter:
                try:
                    content = docling_converter.convert(file_path_str).document.export_to_markdown()
                except Exception:
                    if ext == ".docx":
                        doc = DocxDocument(file_path_str)
                        content = "\n".join(p.text for p in doc.paragraphs if p.text.strip())
                    elif ext in {".xlsx", ".xls"}:
                        wb = openpyxl.load_workbook(file_path_str)
                        content = []
                        for sheet in wb.sheetnames:
                            ws = wb[sheet]
                            content.append(f"\n## Sheet: {sheet}")
                            for row in ws.iter_rows(values_only=True):
                                content.append(" | ".join(str(c) if c else "" for c in row))
                        content = "\n".join(content)
                    else:
                        raise
            else:
                if ext == ".docx":
                    doc = DocxDocument(file_path_str)
                    content = "\n".join(p.text for p in doc.paragraphs if p.text.strip())
                elif ext in {".xlsx", ".xls"}:
                    wb = openpyxl.load_workbook(file_path_str)
                    content = []
                    for sheet in wb.sheetnames:
                        ws = wb[sheet]
                        content.append(f"\n## Sheet: {sheet}")
                        for row in ws.iter_rows(values_only=True):
                            content.append(" | ".join(str(c) if c else "" for c in row))
                    content = "\n".join(content)
                else:
                    raise ValueError(f"Không thể xử lý định dạng {ext} mà không có Docling")

    except Exception as e:
        logger.error(f"Lỗi chuyển đổi: {e}")
        raise

    if not content:
        raise ValueError("Nội dung trống sau khi chuyển đổi")
        
    return content


In [64]:
try:
    print(f"🔄 Đang xử lý file: {input_file_path}")
    raw_md_text = convert_document(input_file_path, output_dir="/tmp")
    
    Path(extracted_markdown_path).write_text(raw_md_text, encoding="utf-8")
    
    print(f"✅ Thành công! Đã lưu file Markdown thô tại: {extracted_markdown_path}")
    print("\n--- PREVIEW ---")
    print(raw_md_text[:500])
    
except Exception as e:
    traceback.print_exc()
    print(f"❌ Có lỗi xảy ra: {e}")

INFO - Phát hiện PDF Text, thử chạy Docling...
INFO - detected formats: [<InputFormat.PDF: 'pdf'>]
INFO - Going to convert document batch...
INFO - Initializing pipeline for StandardPdfPipeline with options hash a6a2eef09bfdbd6bdaeaf15fcaebde59
INFO - rapidocr cannot be used because onnxruntime is not installed.
INFO - easyocr cannot be used because it is not installed.
INFO - Accelerator device: 'cuda:0'
[INFO] 2026-05-18 15:10:15,806 [RapidOCR] base.py:22: Using engine_name: torch
[INFO] 2026-05-18 15:10:15,806 [RapidOCR] device_config.py:64: Using GPU device with ID: 0
[INFO] 2026-05-18 15:10:15,815 [RapidOCR] download_file.py:60: File exists and is valid: /home/trung-ai/miniconda3/envs/ocr_docs/lib/python3.12/site-packages/rapidocr/models/ch_PP-OCRv4_det_mobile.pth
[INFO] 2026-05-18 15:10:15,815 [RapidOCR] main.py:50: Using /home/trung-ai/miniconda3/envs/ocr_docs/lib/python3.12/site-packages/rapidocr/models/ch_PP-OCRv4_det_mobile.pth


🔄 Đang xử lý file: /home/trung-ai/chatbot_hcns/create_documents/doc/241025_E_QUY_TRÌNH_DỰ_ÁN.pdf


[INFO] 2026-05-18 15:10:16,135 [RapidOCR] base.py:22: Using engine_name: torch
[INFO] 2026-05-18 15:10:16,136 [RapidOCR] device_config.py:64: Using GPU device with ID: 0
[INFO] 2026-05-18 15:10:16,137 [RapidOCR] download_file.py:60: File exists and is valid: /home/trung-ai/miniconda3/envs/ocr_docs/lib/python3.12/site-packages/rapidocr/models/ch_ptocr_mobile_v2.0_cls_mobile.pth
[INFO] 2026-05-18 15:10:16,137 [RapidOCR] main.py:50: Using /home/trung-ai/miniconda3/envs/ocr_docs/lib/python3.12/site-packages/rapidocr/models/ch_ptocr_mobile_v2.0_cls_mobile.pth
[INFO] 2026-05-18 15:10:16,177 [RapidOCR] base.py:22: Using engine_name: torch
[INFO] 2026-05-18 15:10:16,178 [RapidOCR] device_config.py:64: Using GPU device with ID: 0
[INFO] 2026-05-18 15:10:16,193 [RapidOCR] download_file.py:60: File exists and is valid: /home/trung-ai/miniconda3/envs/ocr_docs/lib/python3.12/site-packages/rapidocr/models/ch_PP-OCRv4_rec_mobile.pth
[INFO] 2026-05-18 15:10:16,194 [RapidOCR] main.py:50: Using /home/tr

✅ Thành công! Đã lưu file Markdown thô tại: /home/trung-ai/chatbot_hcns/create_documents/md_files/241025_E_QUY_TRÌNH_DỰ_ÁN.md

--- PREVIEW ---
CÔNG TY CỔ PHẦN AIPT VIỆT NAM

E - QUY TRÌNH
TRIỂN KHAI DỰ ÁN

NGÀY HIỆU LỰC: 02-10-2024

#### 1. Mục đích

Quy trình dự án nhằm hướng dẫn CBNV thuộc các phòng ban liên quan phối hợp triển khai công việc nội bộ liên quan tới bán hàng dự án của Công ty Cổ phần AIPT Việt Nam.

#### 2. Phạm vi áp dụng

Áp dụng riêng dành cho Phòng Kinh doanh, Phòng Kỹ thuật, Phòng Xuất nhập khẩu

#### 3. Trách nhiệm

Toàn bộ nhân viên Công ty Cổ phần AIPT Việt Nam chịu trách nhiệm thực hiện theo quy trình này.

###


# BƯỚC 2: CHUẨN HÓA DỮ LIỆU (CLEANING)
Tinh chỉnh bộ lọc Regex tại đây. Nếu có văn bản format lạ (như Phần, Mục, Điều không có dấu chấm...), hãy cập nhật thêm quy tắc vào hàm `clean_ocr_markdown`.

In [65]:
import re

def clean_ocr_markdown(text: str) -> str:
    """
    Hàm dọn dẹp cuối cùng sau khi đã có cấu trúc Markdown chuẩn.
    Tập trung vào sửa lỗi font, khoảng trắng và các ký tự OCR sai.
    """
    # Fix các lỗi xuống dòng thừa thải giữa các dòng văn bản
    text = re.sub(r'(?<=[^\n])\n(?=[a-zâêôư])', ' ', text)
    
    # Fix lỗi OCR phổ biến của tiếng Việt (ví dụ: DIỀU -> ĐIỀU)
    text = re.sub(r'## DIỀU', '## ĐIỀU', text, flags=re.IGNORECASE)
    
    # Chuẩn hóa khoảng trắng
    text = re.sub(r' +', ' ', text)
    text = re.sub(r'\n{3,}', '\n\n', text)
    
    return text.strip()

In [ ]:
# SỬA LẠI LOGICS TRUYỀN FILE: Sử dụng file thô vừa tạo từ BƯỚC 1 để Clean
if not Path(extracted_markdown_path).exists():
    print(f"❌ Lỗi: Không tìm thấy file thô tại {extracted_markdown_path}. Hãy chạy BƯỚC 1 trước.")
else:
    print(f"⏳ Đang làm sạch file: {extracted_markdown_path}...")
    try:
        # 1. Đọc từ file thô (BƯỚC 1)
        with open(extracted_markdown_path, "r", encoding="utf-8") as f:
            raw_text = f.read()

        # 2. Xử lý Smart Clean
        cleaned_text = clean_ocr_markdown(raw_text)

        # 3. Lưu vào file sạch (cleaned_markdown_path từ Cell 3)
        with open(cleaned_markdown_path, "w", encoding="utf-8") as f:
            f.write(cleaned_text)

        print(f"✅ Đã làm sạch và lưu tại: {cleaned_markdown_path}")
        
        # In preview
        print("\n🔍 PREVIEW (CLEANED):")
        print(cleaned_text[:500] + "\n[...]")

    except Exception as e:
        print(f"❌ Lỗi Cleaning: {e}")


⏳ Đang làm sạch file: /home/trung-ai/chatbot_hcns/create_documents/md_files/241025_E_QUY_TRÌNH_DỰ_ÁN.md...
✅ Đã làm sạch và lưu tại: /home/trung-ai/chatbot_hcns/create_documents/md_final_files/241025_E_QUY_TRÌNH_DỰ_ÁN_cleaned.md

🔍 PREVIEW (CLEANED):
CÔNG TY CỔ PHẦN AIPT VIỆT NAM

E - QUY TRÌNH
TRIỂN KHAI DỰ ÁN

NGÀY HIỆU LỰC: 02-10-2024

#### 1. Mục đích

Quy trình dự án nhằm hướng dẫn CBNV thuộc các phòng ban liên quan phối hợp triển khai công việc nội bộ liên quan tới bán hàng dự án của Công ty Cổ phần AIPT Việt Nam.

#### 2. Phạm vi áp dụng

Áp dụng riêng dành cho Phòng Kinh doanh, Phòng Kỹ thuật, Phòng Xuất nhập khẩu

#### 3. Trách nhiệm

Toàn bộ nhân viên Công ty Cổ phần AIPT Việt Nam chịu trách nhiệm thực hiện theo quy trình này.

###
[...]


: 